> 在线讲义：[第 13 章 · 多元微积分与自动微分：从机械臂雅可比到反向传播](https://wikieden.github.io/math-and-eval-stats/lessons/13-calculus-autodiff.html) · 本书《数学与实验统计》

# 第 13 章 · 多元微积分与自动微分

讲义：[13-calculus-autodiff.md](https://wikieden.github.io/math-and-eval-stats/lessons/13-calculus-autodiff.html)。一格一格往下跑；只算讲义里讲过的例子，练习的数自己算。

## 讲义 ↔ 代码

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 | 梯度 | `grad_f(1, 2)` | [8, 3] |
| 第二步 | 雅可比 | `jacobian_2link(30°, 60°)` | [[−1.5, −1.0], [0.866, 0]] |
| 第三步 | Jθ̇ | `J @ [0.1, -0.2]` | (0.05, 0.0866) |
| 第四步 | 反向模式 | `Var + backward()` | ∂f/∂x = 2.5839，∂f/∂y = 2 |
| 第五步 | 中心差分 | `numeric_jacobian` | 误差 ~1e-10 |

## 定义

In [ ]:
import math
import numpy as np

`grad_f`：第一步：f(x, y) = x² + 3xy 的梯度 (2x + 3y, 3x)。

In [ ]:
def grad_f(x: float, y: float) -> np.ndarray:
    """第一步：f(x, y) = x² + 3xy 的梯度 (2x + 3y, 3x)。"""
    return np.array([2 * x + 3 * y, 3 * x])

`fk_2link`：第二步：两连杆平面臂的正运动学，末端 (x, y)。

In [ ]:
def fk_2link(t1: float, t2: float, l1: float = 1.0, l2: float = 1.0) -> np.ndarray:
    """第二步：两连杆平面臂的正运动学，末端 (x, y)。"""
    return np.array([l1 * math.cos(t1) + l2 * math.cos(t1 + t2), l1 * math.sin(t1) + l2 * math.sin(t1 + t2)])

`jacobian_2link`：第二步：解析雅可比 J = ∂(x, y)/∂(θ1, θ2)。

In [ ]:
def jacobian_2link(t1: float, t2: float, l1: float = 1.0, l2: float = 1.0) -> np.ndarray:
    """第二步：解析雅可比 J = ∂(x, y)/∂(θ1, θ2)。"""
    s1, c1, s12, c12 = math.sin(t1), math.cos(t1), math.sin(t1 + t2), math.cos(t1 + t2)
    return np.array([[-l1 * s1 - l2 * s12, -l2 * s12], [l1 * c1 + l2 * c12, l2 * c12]])

`numeric_jacobian`：第五步：中心差分 (f(x+h e_i) - f(x-h e_i)) / 2h，用来核对解析 / 自动微分。

In [ ]:
def numeric_jacobian(f, x: np.ndarray, h: float = 1e-6) -> np.ndarray:
    """第五步：中心差分 (f(x+h e_i) - f(x-h e_i)) / 2h，用来核对解析 / 自动微分。"""
    cols = []
    for i in range(len(x)):
        e = np.zeros_like(x)
        e[i] = h
        cols.append((f(*(x + e)) - f(*(x - e))) / (2 * h))
    return np.stack(cols, axis=1)

`Var`：第四步：50 行以内的标量反向模式自动微分（只支持 +、×、sin）。

In [ ]:
class Var:
    """第四步：50 行以内的标量反向模式自动微分（只支持 +、×、sin）。"""

    def __init__(self, value, parents=()):
        self.value, self.parents, self.grad = value, parents, 0.0

    def __add__(self, other):
        return Var(self.value + other.value, ((self, 1.0), (other, 1.0)))

    def __mul__(self, other):
        return Var(self.value * other.value, ((self, other.value), (other, self.value)))

    def sin(self):
        return Var(math.sin(self.value), ((self, math.cos(self.value)),))

    def backward(self):
        order, seen = [], set()

        def topo(v):  # 拓扑排序：保证一个节点的伴随量收齐了才往回传
            if id(v) not in seen:
                seen.add(id(v))
                for p, _ in v.parents:
                    topo(p)
                order.append(v)

        topo(self)
        self.grad = 1.0
        for v in reversed(order):
            for p, local in v.parents:
                p.grad += v.grad * local  # 伴随量 = 上游伴随 × 局部导数，多条路径相加

## 一步一步跑

### 第一步 梯度

In [ ]:
grad_f(1, 2)

### 第二步 正运动学与雅可比

In [ ]:
T = (math.radians(30), math.radians(60)); J = jacobian_2link(*T); fk_2link(*T), J, np.linalg.det(J)

### 第三步 末端速度

In [ ]:
J @ np.array([0.1, -0.2])

### 第四步 反向模式

In [ ]:
x, y = Var(2.0), Var(3.0); f = x * y + x.sin(); f.backward(); f.value, x.grad, y.grad

### 第五步 数值核对

In [ ]:
np.abs(numeric_jacobian(fk_2link, np.array(T)) - J).max()

## 代码里才看得到的

1. 反向传播前要拓扑排序，共享节点的伴随量收齐了才往回传。
2. p.grad += 而不是 =：多条路径的贡献相加。
3. 中心差分要用 float64。

## 自己改一改

- 给 Var 加 exp
- θ2 从 60° 减到 1°，看 det J

In [ ]:
# 在这里改参数试试


## 口诀

> 偏导拼成梯度，指向最陡上坡；
> 雅可比是导数表，行列为零就奇异；
> 链式法则矩阵乘，关节速度乘出末端；
> 一个 loss 反向推，求导记得差分核。